## Učitavanje biblioteka i podataka

In [13]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib as plt
import plotly
import spacy
import nltk
import re
import string
import srbai

In [18]:
# Function for importing .xml data as a data frame
import os,glob
import pandas as pd

def load_data(path):
  texts = []
  for filename in glob.glob(os.path.join(path, "*.txt")):
    with open(filename, 'r') as f:
      texts.append(f.read())
      df = pd.DataFrame({'texts' : texts})

  return df

In [19]:
# Importing data
data_path = "/Users/teamihajlov/Projects/CLIB_TopicVisualization/data"
data = load_data(data_path)
data.head()

,texts
0,"већда\nпотражити\nтенџера\n,\nи\nизмећарка\nВа..."
1,"странапогледати\n,\n„\nкад\nдоћи\nовамо\n.\n“\..."
2,протадозвола\nда\nтамо\nи\nостати\n.\nето\nтај...
3,",дакле\n,\nонај\nтребати\nда\nбити\nуправитељ\..."
4,одпрозор\nне\nмоћи\nда\nдогледати\nместо\nгде\...


In [20]:
data.shape

(678, 1)

In [21]:
# Function for reading .txt data
def covnert_path_to_list(path):
  words = []
  with open(path, 'r') as f:
    words = (f.read())
    words_list = list(words.split('\n'))

    return words_list

# Importing stopwords for Serbian
stopwords_path = '/content/CLIB_TopicVisualization/stopwordsSRB_cyr.txt'
stopwords = covnert_path_to_list(stopwords_path)

# Importing corpus-specific stopwords
stops_add_path = '/content/CLIB_TopicVisualization/stops_extra.txt'
stopwords_add = covnert_path_to_list(stops_add_path)

print(stopwords)
print(len(stopwords))
print(stopwords_add)
print(len(stopwords_add))

FileNotFoundError: [Errno 2] No such file or directory: '/content/CLIB_TopicVisualization/stopwordsSRB_cyr.txt'

In [ ]:
# Merging Serbian and corpus-specific stopwords
stopwords.extend(stopwords_add)
print(stopwords)
print(len(stopwords))

In [ ]:
# Converting final stopwords to latin script
stopwords_lat = preslovi(stopwords)
print(stopwords_lat)
print(len(stopwords_lat))

In [ ]:
file_path = "stopwords_lat.txt"
with open(file_path, 'w') as file:
    # Join the list elements into a single string with a newline character
    data_to_write = '\n'.join(stopwords_lat)

    # Write the data to the file
    file.write(data_to_write)

## Priprema podataka

In [ ]:
# Defining functions for the preprocessing pipeline


# Defining clean_text funtion for removing special characters, punctuation, numbers, and stopwords from text
# importing libraries


# defining a clean_text function for removing special characters, and converting the text to lowercase

def clean_text(text):
      cleaned_text = []
      for t in text:
        # removing extra whitespace and special characters
        t = t.replace("\n\n", " ").replace('\n', ' ').replace('—', '').replace('„', '').replace('“','').replace('«', '').replace('»', '').replace('@card@', '').replace('’', '').replace('–', '').replace('\"', '')
        t = t.lower() # coverting text to lowercase
     #   t = t.translate(str.maketrans(" ", " ", string.punctuation)) # removing puntuation
        t = re.sub(r'\b\w{1,2}\b', '', t)
      #  t = " ".join([word for word in t.split() if word not in stopwords])
      #  t = "".join([word for word in t if not word.isdigit()]) # removing numbers
        t = t.strip() # removing extra whitespace from beginning and end of a string
        t = re.sub(" +", " ", t) # removing extra whitespace after each word
        if t:
          cleaned_text.append(t) # appending normalized text to a new list

      return(cleaned_text) # returning normalized texts

# defining a convert_to_lat function for converting text from cyrilic to latin script

def convert_to_lat(texts):
  texts_lat = []
  for text in texts:
    lat = preslovi(text)
    texts_lat.append(lat)

  return texts_lat

# defining a lematize_text function for text lematization

def lematize_text(text):
    final = []
    for sent in text:
      # tokenization
      tok_sentence  = tokenizuj(sent)
      # comparing tokens with lexicon
      mag_sentence = magija(tok_sentence , lexicon)
      # tagging
      tag_sentence = tagiraj(mag_sentence, model)
      tag_sentence_lema = lematizuj(tag_sentence, recnik_lema)
      sent_lema = []
      sent_lema.append(tag_sentence_lema)
      # extracting lemmas
      lemmas = []
      for sent in sent_lema:
        for token in sent:
          lema = token.split('\t')[2]
          lemmas.append(''.join(lema))

      final.append(lemmas)

    return final

# defining a join_tokens function to join tokens into sentences

def join_tokens(texts):
    final = []

    for text in texts:
        joined_text = ""
        for token in text:
            if token in ['.', ',', '!', '?']:  # If the token is punctuation, don't add a space before it
                joined_text = joined_text.rstrip() + token
            else:
                joined_text += token + " "

        final.append([joined_text.strip()])

    return final

# defining a function for removing stopwords and numbers from text

def remove_stopwords(texts, stopwords = stopwords_lat):
    final = []

    for text in texts:
        clean_texts = []
        for t in text:
            # Remove stopwords
            t = " ".join([word for word in t.split() if word.lower() not in stopwords])
            # Remove numbers
            t = " ".join([word for word in t.split() if not word.isdigit()])
            # Replace punctuation with spaces, then remove punctuation
            t = t.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
            t = t.translate(str.maketrans("", "", string.punctuation))
            # Remove extra whitespace from beginning and end of string
            t = t.strip()
            # Remove extra whitespace between words
            t = re.sub(" +", " ", t)
            if t:
                clean_texts.append(t)  # Append normalized text to the list
        final.append(clean_texts)  # Append the processed list to final

    return final

In [ ]:
# Creating a pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

preprocessing_pipeline = Pipeline(steps=[
    ('clean_text', FunctionTransformer(clean_text)),
    ('convert_to_lat', FunctionTransformer(convert_to_lat)),
    ('lematize_text', FunctionTransformer(lematize_text)),
    ('join_tokens', FunctionTransformer(join_tokens)),
    ('remove_stopwords', FunctionTransformer(remove_stopwords))])

In [ ]:
data_clean = preprocessing_pipeline.fit_transform(data['texts'])

In [ ]:
print(data_clean[0])
print(len(data_clean))

In [ ]:
data.drop(columns = ['texts_clean'], inplace = True)

In [ ]:
data.head()

In [ ]:
df = pd.DataFrame(data_clean, columns = ['texts_clean'])

In [ ]:
merged_df = pd.merge(data, df, left_index=True, right_index=True)
merged_df.head()

In [ ]:
merged_df.to_csv('merged_df.csv')

Modeli prebaceni u novu svesku.